# Plant Disease Detection — training notebook (PlantVillage, 38 classes)

Transfer learning with **MobileNetV2** (ImageNet) — see `decisions/0002` and `decisions/0003`.

**Run on Google Colab with a GPU:** *Runtime → Change runtime type → T4 GPU*. Total time ≈ 30–45 min.

Outputs (downloaded at the end, copy them into `backend/models/`):
- `plant_disease_v1.keras` — model with preprocessing (`Rescaling` to [-1, 1]) **embedded**; it expects raw 0–255 RGB 224×224 input
- `class_names.json` — output index → label
- `metrics.json`, `classification_report.txt`, `confusion_matrix.png` — copy into `ml/reports/`

⚠️ The backend container uses **TensorFlow 2.19** (`backend/requirements.txt`). Keep the training TF version
the same major/minor if possible (cell 1 prints it) so the `.keras` file loads without issues.

In [ ]:
import tensorflow as tf, keras, sys
print("TensorFlow", tf.__version__, "| Keras", keras.__version__, "| Python", sys.version.split()[0])
print("GPU:", tf.config.list_physical_devices("GPU"))
# If the TF version differs from the backend's (2.19), you can align it, then restart the runtime:
# !pip install -q "tensorflow==2.19.*"

In [ ]:
import json, os
import numpy as np
import tensorflow_datasets as tfds

IMG_SIZE = 224
BATCH = 64
SEED = 42
AUTOTUNE = tf.data.AUTOTUNE

# PlantVillage only has a 'train' split in TFDS → carve out 80/10/10 deterministically
(train_raw, val_raw, test_raw), info = tfds.load(
    "plant_village", split=["train[:80%]", "train[80%:90%]", "train[90%:]"],
    as_supervised=True, with_info=True, shuffle_files=False,
)
CLASS_NAMES = info.features["label"].names
NUM_CLASSES = len(CLASS_NAMES)
print(NUM_CLASSES, "classes;", info.splits["train"].num_examples, "images")

In [ ]:
def resize(image, label):
    # Keep raw 0-255 pixel values — normalisation lives inside the model
    return tf.image.resize(image, (IMG_SIZE, IMG_SIZE)), label

augment = keras.Sequential([
    keras.layers.RandomFlip("horizontal_and_vertical"),
    keras.layers.RandomRotation(0.15),
    keras.layers.RandomZoom(0.15),
    keras.layers.RandomContrast(0.15),
], name="augment")

train_ds = (train_raw.shuffle(4096, seed=SEED).map(resize, num_parallel_calls=AUTOTUNE).batch(BATCH)
            .map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))
val_ds = val_raw.map(resize, num_parallel_calls=AUTOTUNE).batch(BATCH).cache().prefetch(AUTOTUNE)
test_ds = test_raw.map(resize, num_parallel_calls=AUTOTUNE).batch(BATCH).prefetch(AUTOTUNE)

## Model: MobileNetV2 backbone + classification head (augmentation is kept OUT of the saved model)

In [ ]:
base = keras.applications.MobileNetV2(input_shape=(IMG_SIZE, IMG_SIZE, 3), include_top=False, weights="imagenet")
base.trainable = False

inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name="image")
x = keras.layers.Rescaling(1 / 127.5, offset=-1, name="mobilenet_preprocess")(inputs)  # == mobilenet_v2.preprocess_input
x = base(x, training=False)  # keep BatchNorm in inference mode, also while fine-tuning
x = keras.layers.GlobalAveragePooling2D()(x)
x = keras.layers.Dropout(0.3)(x)
outputs = keras.layers.Dense(NUM_CLASSES, activation="softmax", name="predictions")(x)
model = keras.Model(inputs, outputs, name="plant_disease_mobilenetv2")
model.summary()

## Stage 1 — train the head (backbone frozen)

In [ ]:
callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_accuracy", patience=3, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=2),
]
model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=8, callbacks=callbacks)

## Stage 2 — fine-tune the top of the backbone with a low learning rate

In [ ]:
base.trainable = True
for layer in base.layers[:-40]:
    layer.trainable = False
model.compile(optimizer=keras.optimizers.Adam(1e-5), loss="sparse_categorical_crossentropy", metrics=["accuracy"])
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

## Evaluate on the held-out test split

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

test_loss, test_acc = model.evaluate(test_ds)
y_true = np.concatenate([y.numpy() for _, y in test_ds])
y_pred = np.argmax(model.predict(test_ds), axis=1)

report = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print(report)
print(f"Test accuracy: {test_acc:.4f}  (target ≥ 0.95)")

cm = confusion_matrix(y_true, y_pred, normalize="true")
fig, ax = plt.subplots(figsize=(16, 14))
im = ax.imshow(cm, cmap="Greens")
ax.set_xticks(range(NUM_CLASSES)); ax.set_yticks(range(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=7); ax.set_yticklabels(CLASS_NAMES, fontsize=7)
ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(f"Normalised confusion matrix — test acc {test_acc:.4f}")
fig.colorbar(im); fig.tight_layout(); fig.savefig("confusion_matrix.png", dpi=120)

## Export artefacts

In [ ]:
model.save("plant_disease_v1.keras")
with open("class_names.json", "w") as f:
    json.dump(CLASS_NAMES, f, indent=2)
with open("classification_report.txt", "w") as f:
    f.write(report)
with open("metrics.json", "w") as f:
    json.dump({"model": "plant_disease_v1", "backbone": "MobileNetV2", "img_size": IMG_SIZE,
               "num_classes": NUM_CLASSES, "test_accuracy": float(test_acc), "test_loss": float(test_loss),
               "tensorflow": tf.__version__, "split": "tfds plant_village train[:80%]/[80%:90%]/[90%:]"}, f, indent=2)

# Sanity check: reload and predict one raw (0-255) test image, exactly like the backend does
reloaded = keras.models.load_model("plant_disease_v1.keras", compile=False)
img, label = next(iter(test_raw.map(resize).batch(1)))
print("true:", CLASS_NAMES[int(label[0])], "| predicted:", CLASS_NAMES[int(np.argmax(reloaded(img)))])

In [ ]:
# Download the files (Colab). Then on your machine:
#   mv plant_disease_v1.keras class_names.json  <repo>/DetectionSystem/backend/models/
#   mv metrics.json classification_report.txt confusion_matrix.png  <repo>/DetectionSystem/ml/reports/
from google.colab import files
for name in ["plant_disease_v1.keras", "class_names.json", "metrics.json", "classification_report.txt", "confusion_matrix.png"]:
    files.download(name)